# WeebCentral → Telegram Leech (Colab)

1. Cell 1 — install
2. Cell 2 — config
3. Cell 3 — helpers
4. Cell 4 — **full download + Telegram post**
5. Cell 5 — optional Drive


In [ ]:
# ══ CELL 1 — Install + clone
!pip install -q requests 'httpx[http2]' nest_asyncio beautifulsoup4 lxml Pillow fpdf2 tqdm pyrofork TgCrypto aiofiles
import os
os.chdir('/content')
!rm -rf /content/weebcentral_downloader
!git clone --depth 1 https://github.com/Yui007/weebcentral_downloader.git
os.makedirs('/content/weebcentral_telegram_leech', exist_ok=True)
print('OK')


In [ ]:
# ══ CELL 2 — Config
try:
    from google.colab import userdata
    BOT_TOKEN = userdata.get("BOT_TOKEN") or ""
    API_ID = int(userdata.get("API_ID") or 0)
    API_HASH = userdata.get("API_HASH") or ""
    USER_ID = int(userdata.get("USER_ID") or 0)
    DUMP_CHANNEL = int(userdata.get("DUMP_CHANNEL") or 0)
    UPLOAD_CHANNEL = int(userdata.get("UPLOAD_CHANNEL") or DUMP_CHANNEL or 0)
except Exception:
    BOT_TOKEN = ""
    API_ID = 0
    API_HASH = ""
    USER_ID = 0
    DUMP_CHANNEL = -1001234567890
    UPLOAD_CHANNEL = -1001234567890

CHANNEL_LINK = "https://t.me/Asa_Mikata373"
OUTPUT_DIR = "/content/manga"
OUTPUT_FORMAT = "pdf"
assert BOT_TOKEN and API_ID and API_HASH and USER_ID
assert DUMP_CHANNEL
print("Config OK")


In [ ]:
# ══ CELL 3 — Title fix + telegram_leech helper
import urllib.request
from pathlib import Path
Path("/content/weebcentral_telegram_leech").mkdir(parents=True, exist_ok=True)
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/zenin-373/Manga-DL/main/colab/scrape_chapter_list_fixed.py",
    "/content/weebcentral_telegram_leech/scrape_chapter_list_fixed.py",
)
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/zenin-373/Manga-DL/main/colab/telegram_leech.py",
    "/content/weebcentral_telegram_leech/telegram_leech.py",
)
print("Helpers downloaded (title fix + telegram_leech)")


In [ ]:
# ══ CELL 4 — Scrape → download → Telegram leech (FULL)
import sys, asyncio, re
from pathlib import Path

sys.path.insert(0, "/content/weebcentral_downloader/colab")
sys.path.insert(0, "/content/weebcentral_telegram_leech")

from colab_scraper import scrape_manga_info
from scrape_chapter_list_fixed import scrape_chapter_list
from colab_downloader import parse_chapter_selection, download_chapters, sanitize
from telegram_leech import TelegramLeech, download_cover

import nest_asyncio
nest_asyncio.apply()

SERIES_URL = input(
    "WeebCentral series URL\n"
    "   e.g. https://weebcentral.com/series/01J.../slug\n"
    "   > "
).strip()

manga_info = scrape_manga_info(SERIES_URL)
chapters = scrape_chapter_list(SERIES_URL)
total = len(chapters)

print(f"\n{'='*62}")
print(f"  {'#':>4}   {'Chapter':<50}  Date")
print(f"{'='*62}")
for ch in chapters:
    print(f"  {str(ch['index']).rjust(4)}   {ch['title'][:50].ljust(50)}  {ch['date']}")
print(f"{'='*62}\n  Total: {total}\n")

print("Selection: all | single 5 | range 1-10 | 1,5,9")
while True:
    sel_input = input(f"Select chapters (1–{total}): ").strip()
    try:
        selected = parse_chapter_selection(sel_input, total)
        print(f"   {len(selected)} chapter(s)")
        break
    except ValueError as e:
        print(f"   {e}")

series_dir = download_chapters(
    manga_info=manga_info,
    chapters=chapters,
    selected_indices=selected,
    output_format=OUTPUT_FORMAT,
    output_dir=OUTPUT_DIR,
)
print(f"Files in: {series_dir}")

async def leech_to_telegram():
    leech = TelegramLeech(
        api_id=API_ID,
        api_hash=API_HASH,
        bot_token=BOT_TOKEN,
        user_id=USER_ID,
        dump_channel=DUMP_CHANNEL,
        upload_channel=UPLOAD_CHANNEL or DUMP_CHANNEL,
        channel_link=CHANNEL_LINK,
    )
    await leech.start()

    cover_path = None
    if manga_info.get("cover_url"):
        cover_path = download_cover(manga_info["cover_url"], Path(series_dir) / "cover.jpg")
    await leech.post_series_poster(manga_info, total, cover_path)

    pdfs = sorted(Path(series_dir).glob("*.pdf"))
    print(f"Found {len(pdfs)} PDF(s)")

    by_num = {}
    for p in pdfs:
        m = re.search(r"Ch0*(\d+)", p.name)
        if m:
            by_num[int(m.group(1))] = p

    ok = 0
    for idx in selected:
        ch_num = idx + 1
        ch = chapters[idx]
        pdf = by_num.get(ch_num)
        if not pdf:
            frag = sanitize(ch["title"])[:40]
            for p in pdfs:
                if frag and frag in p.name:
                    pdf = p
                    break
        if not pdf:
            print(f"No PDF for chapter {ch_num}")
            continue
        print(f"Uploading Ch {ch_num}: {pdf.name}")
        if await leech.upload_chapter_pdf(
            str(pdf),
            manga_info["title"],
            ch.get("number") or ch_num,
            chapter_title=ch.get("title") or "",
            series_url=SERIES_URL,
            thumb_path=cover_path,
        ):
            ok += 1
        await asyncio.sleep(1.5)

    await leech.notify(f"Done {ok}/{len(selected)} — {manga_info['title']}")
    await leech.stop()
    print(f"Uploaded {ok}/{len(selected)}")

await leech_to_telegram()


In [ ]:
# ══ CELL 5 — Optional Drive
from google.colab import drive
import shutil, re
drive.mount('/content/drive')
zip_stem = re.sub(r'[\\/:*?"<>|]', '_', manga_info['title']).strip() + '_manga'
drive_out = f'/content/drive/MyDrive/{zip_stem}'
shutil.copytree(series_dir, drive_out, dirs_exist_ok=True)
print('Saved to', drive_out)
